In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
original_path = Path(r"C:\Users\bouba.ismalia\Stichting Hogeschool Utrecht\FCA-DA-P - data\processed\clean_original.csv")

# Let pandas detect the delimiter
data = pd.read_csv(original_path, sep=None, engine="python", encoding="utf-8-sig")

# Drop unnamed columns
data = data.loc[:, ~data.columns.str.contains('^Unnamed')]

print(data.shape)
data


In [ ]:
data.columns

# Handle low frequency categories in columns

sdo5_degree_degree

sdo1_previous_Previous_Education_Type

sdo2_skc_ADVIES_DEF

  (see cleaning data script for detalis)

In [ ]:
# -----------------------------------------------------------------------------
# Handling Categorical Variables for Mixed Model Families
#
# Purpose
#   Apply a frequency cutoff to merge rare categories into "Other" for selected
#   categorical columns. This reduces noise, prevents overfitting from tiny
#   groups, and avoids one-hot encoding explosion while keeping features
#   compatible across models (LR, SVM → one-hot later; RF → benefits from lower
#   cardinality).
#
# Cutoff and Calculation
#   Chosen cutoff: 0.5% of the dataset (p = 0.005).
#   For this dataset:
#       N = 47,953 rows
#       minimum_count = N × p = 47,953 × 0.005 ≈ 240
#   Any category with < ~240 rows is replaced by "Other".
#
# Why 0.5%?
#   • Cross-validation stability: in 5-fold CV, ~240 total ⇒ ~48 per fold,
#     sufficient for reliable impurity/gain estimation and to avoid empty/micro
#     leaves.
#   • Bias-variance balance: removes idiosyncratic, near-unique levels that
#     encourage overfitting while retaining meaningful categories.
#   • Computational efficiency: limits effective cardinality, speeding up model
#     fitting and preventing one-hot feature explosion in linear/SVM pipelines.
#   • Pragmatic middle ground: lower cutoffs (e.g., 0.1%) admit many unstable
#     levels; higher (e.g., 1%) may discard useful detail.
#
# Notes
#   • NaN values are preserved (not merged into "Other").
#   • Only the specified columns are processed.
#   • If CV folds change, the cutoff can be recalculated as:
#       p = (min_per_fold * n_folds) / N
# -----------------------------------------------------------------------------


# Columns to apply the cutoff to
cat_cols = [
    "sdo5_degree_degree",
    "sdo1_previous_Previous_Education_Type",
    "sdo2_skc_ADVIES_DEF",
]

N = 47953
p = 0.005
min_count = ceil(N * p)  # ≈ 240

for col in cat_cols:
    vc = data[col].value_counts(dropna=False)
    # Exclude NaN from replacement logic; only merge labeled rare levels
    rare_levels = [lvl for lvl in vc.index if (not pd.isna(lvl)) and (vc[lvl] < min_count)]
    if rare_levels:
        data[col] = data[col].where(~data[col].isin(rare_levels), "Others")


In [ ]:
print(data['sdo5_degree_degree'].value_counts())

In [ ]:
print(data['sdo1_previous_Previous_Education_Type'].value_counts())

In [ ]:
print(data['sdo2_skc_ADVIES_DEF'].value_counts())

# Missing Values Imputation Strategy

In [ ]:
# NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
data.info()

In [ ]:
# -----------------------------------------------------------------------------
# Missing Value Imputation Plan — with Missingness Overview
#
# Context:
#   - Mixed data types (numeric, categorical/object, dates stored as object).
#   - “Datum” in Dutch = date. Date fields stored as object should be parsed.
#   - Goal: produce a complete, model-ready table.
#
# Dataset size:
#   N = 47,953 rows
#
# Missingness summary (by column group)
#   Orientation participation (structural, numeric):
#     • sdo2_orientation_Number_of_Events_Attended ............ 38,727  (80.76%)
#     • sdo2_orientation_Number_of_Event_Types ................ 38,727  (80.76%)
#
#   Academic performance (numeric):
#     • sdo6_results_Average_Grade_B .......................... 11,543  (24.07%)
#     • sdo6_results_Average_Grade_A ..........................  7,592  (15.83%)
#     • sdo6_results_Percentage_Credits_B .....................  6,428  (13.40%)
#     • sdo6_results_Potential_Credits_B ......................  6,028  (12.57%)
#     • sdo6_results_Percentage_Credits_A .....................  4,319  ( 9.01%)
#     • sdo6_results_Potential_Credits_A ......................  4,158  ( 8.67%)
#
#   Distances & related postcodes (numeric):
#     • sdo1_prev_distance_distance_to_3584_CS ................  6,462  (13.48%)
#     • sdo1_prev_distance_postcode ...........................  4,538  ( 9.46%)
#     • sdo1_postal_distance_distance_to_3584_CS ..............  3,355  ( 7.00%)
#     • sdo1_postal_distance_postcode .........................  1,536  ( 3.20%)
#
#   Demographics (numeric):
#     • sdo1_characteristics_Dutch_nationality ................  3,476  ( 7.25%)  [float64]
#     • sdo1_characteristics_age_start_study ..................  3,476  ( 7.25%)
#
#   Dates / Datum (object → datetime):
#     • sdo2_skc_SKC_DATUM ....................................  5,248  (10.94%)
#     • sdo1_previous_Final_Exam_Date .........................    779  ( 1.62%)
#     • sdo5_degree_enrollment_date / degree_start_date / degree_end_date: 0% missing
#
# Imputation policy (reasoned by type & missingness)
#   1) Orientation counts (≈81% missing, structural):
#        NaN → 0  (interpretation: no participation)
#        + add <col>_missing_flag ∈ {0,1} to preserve missingness signal.
#
#   2) Academic performance, distances, age (≈7–24% missing, numeric):
#        NaN → median of the column (robust central imputation)
#        + add <col>_missing_flag ∈ {0,1}.
#
#   3) Dutch nationality (float64; likely binary/ordinal encoded):
#        NaN → mode (most frequent value)  + add missing flag.
#        (Keep numeric type to remain compatible with LR/SVM without extra encoding.)
#
#   4) Date/Datum columns (object → datetime):
#        Parse to datetime (errors='coerce'), extract YEAR as integer feature.
#        NaT → YEAR 0 (0 denotes unknown/unavailable year). Keep original datetime if needed.
#
#   5) Categorical objects with no missing (e.g., sdo5_degree_degree, previous education type,
#      SKC advice after earlier preprocessing): no imputation required here.
#
# Rationale for model families:
#   • LR/SVM require fully numeric matrices (no NaN) → above rules ensure completeness,
#     with flags to allow linear models to leverage missingness patterns.
#   • RF benefits from reduced noise and explicit missingness indicators while avoiding NaN.
#   • Year extraction from Datum adds usable temporal signal without NaT issues.
# -----------------------------------------------------------------------------

In [ ]:
# --- A) Orientation columns: high, likely-structural missingness -> 0 + flag ---
orientation_cols = [
    "sdo2_orientation_Number_of_Events_Attended",
    "sdo2_orientation_Number_of_Event_Types",
]
for col in orientation_cols:
    data[f"{col}_missing_flag"] = data[col].isna().astype(int)
    data[col] = data[col].fillna(0)

# --- B) Academic performance, distances, age: median + flag ---
numeric_median_cols = [
    # Academic performance
    "sdo6_results_Average_Grade_A", "sdo6_results_Average_Grade_B",
    "sdo6_results_Percentage_Credits_A", "sdo6_results_Percentage_Credits_B",
    "sdo6_results_Potential_Credits_A", "sdo6_results_Potential_Credits_B",
    # Distances
    "sdo1_prev_distance_distance_to_3584_CS", "sdo1_postal_distance_distance_to_3584_CS",
    # Demographics (continuous)
    "sdo1_characteristics_age_start_study",
]
for col in numeric_median_cols:
    data[f"{col}_missing_flag"] = data[col].isna().astype(int)
    data[col] = data[col].fillna(data[col].median())

# --- C) Dutch nationality (float64): mode + flag (keep numeric encoding) ---
nat_col = "sdo1_characteristics_Dutch_nationality"
if nat_col in data.columns:
    data[f"{nat_col}_missing_flag"] = data[nat_col].isna().astype(int)
    # Compute mode safely (dropna) and fallback to 0 if empty
    if data[nat_col].dropna().shape[0] > 0:
        nat_mode = data[nat_col].dropna().mode()
        nat_fill_value = nat_mode.iloc[0] if len(nat_mode) else 0
    else:
        nat_fill_value = 0
    data[nat_col] = data[nat_col].fillna(nat_fill_value)

# --- D) Date / Datum columns: parse → extract YEAR (int), fill missing YEAR with 0 ---
date_cols = [
    "sdo5_degree_enrollment_date",
    "sdo5_degree_degree_start_date",
    "sdo5_degree_degree_end_date",
    "sdo1_previous_Final_Exam_Date",
    "sdo2_skc_SKC_DATUM",
]
for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors="coerce")  # 'Datum' parsing
    data[f"{col}_year"] = data[col].dt.year.fillna(0).astype(int)

# Optional: if original datetime cols are not needed downstream, they can be dropped/commented
# data.drop(columns=date_cols, inplace=True)
# -----------------------------------------------------------------------------

In [ ]:
data

In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
# -----------------------------------------------------------------------------
# Final Missing Value Cleanup — Post-Imputation Validation
#
# Purpose:
#   Ensure that no missing values remain after the main imputation step.
#   This block finalizes the dataset by addressing any residual NaN values
#   in identifier or non-feature columns and removing redundant date fields.
#
# Summary of results before this step:
#   • All numeric and float64 columns have 0 missing values.
#   • All *_missing_flag columns are complete (0/1).
#   • All *_year columns (derived from Datum/date fields) are complete.
#   • Only a few non-critical columns still contain NaN values:
#       - sdo1_prev_distance_postcode .......... 4,538 missing
#       - sdo1_postal_distance_postcode ........ 1,536 missing
#       - sdo2_skc_SKC_DATUM ................... 5,248 missing (object type)
#
# Actions performed:
#   1) Replace NaN in postcode fields with 0.
#        → These are identifiers, not predictive numeric features;
#          filling with 0 prevents model errors without changing meaning.
#
#   2) Drop original object-type date column (sdo2_skc_SKC_DATUM),
#      since its numeric year-based feature (sdo2_skc_SKC_DATUM_year)
#      is already present and complete.
# -----------------------------------------------------------------------------

fill_0_cols = [
    "sdo1_prev_distance_postcode",
    "sdo1_postal_distance_postcode"
]
data[fill_0_cols] = data[fill_0_cols].fillna(0)

# Drop redundant object-type date column (its numeric year version is retained)
data.drop(columns=["sdo2_skc_SKC_DATUM"], inplace=True)


In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
# -----------------------------------------------------------------------------
# Impute missing values in 'sdo1_previous_Final_Exam_Date' with the mode
#
# Purpose:
#   Retain the raw Datum field and replace missing entries using the most
#   frequently occurring value (mode), preserving datetime dtype.
#
# Notes:
#   • Parses to datetime with errors coerced to NaT.
#   • Computes the mode over non-missing values; if multiple modes exist,
#     uses the earliest (deterministic choice).
#   • Fills 779 missing entries with the selected mode value.
# -----------------------------------------------------------------------------

col = "sdo1_previous_Final_Exam_Date"

# Ensure datetime dtype
data[col] = pd.to_datetime(data[col], errors="coerce")

# Compute mode over valid timestamps
_mode_series = data[col].dropna().mode()

if not _mode_series.empty:
    # If multiple modes, select the earliest deterministically
    fill_value = _mode_series.min()
    data[col] = data[col].fillna(fill_value)
else:
    # Fallback (should not trigger in this dataset): use a neutral sentinel
    # This line can be removed if a fallback is not desired.
    fill_value = pd.Timestamp("1900-01-01")
    data[col] = data[col].fillna(fill_value)


In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

# Feature Derivation

In [ ]:
data.columns

In [ ]:
# Save version 2 data here to later combine it with version 1
data.to_csv(os.path.join("..", "data", "processed", "clean_original.csv"), header=True)